# Soro-Kev on Kaggle 2x T4

Pointer model for tool routing, argument repair, missing parameters, and mutation detection.
One prefill. No token generation. Loss is RLCD: cross-entropy + 0.25 Brier + 0.30 permutation JSD, plus the three auxiliary heads.

**Accelerator:** Notebook settings, GPU, **T4 x2**. Internet **On** so Hugging Face can download `Qwen/Qwen3-0.6B`.

T4 is 16 GB, compute capability 7.5, fp16 only. This profile refuses a 4B backbone.

Upload the training code and the three jsonl files. You do not need `third_party/` or `web/` on Kaggle. The jsonl files are already built.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'gpus', torch.cuda.device_count())
if torch.cuda.is_available():
    print('capability', torch.cuda.get_device_capability(0), 'bf16', torch.cuda.is_bf16_supported())

In [ ]:
!pip install -q -U transformers peft accelerate pyarrow numpy
!pip uninstall -y torchao

In [ ]:
import os
from pathlib import Path

roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('.')]
found = list(Path('/kaggle/input').rglob('train.jsonl')) if Path('/kaggle/input').exists() else []
found += list(Path('.').rglob('train.jsonl'))
if not found:
    raise SystemExit('train.jsonl not found. Add the dataset files to the notebook.')
DATA = found[0].parent
VEYRA = None
for p in [DATA, *DATA.parents]:
    if (p / 'python' / 'src' / 'soro').exists() and (p / 'training' / 'train_soro_kev.py').exists():
        VEYRA = p
        break
if VEYRA is None:
    raise SystemExit('Could not find veyra/python/src/soro next to the jsonl files.')
os.chdir(VEYRA)
os.environ['PYTHONPATH'] = str(VEYRA / 'python' / 'src')
print('VEYRA', VEYRA)
print('DATA', DATA)
for name in ['train.jsonl', 'val_zeroshot.jsonl', 'val_external_human.jsonl']:
    fp = DATA / name
    print(f'{name}: {fp.stat().st_size/1e6:.1f} MB, exists={fp.exists()}')

## Train

`torchrun` uses both T4s. Each GPU holds Qwen3-0.6B in fp16 plus LoRA. Batch size 4 and grad accumulation 2 fit in 16 GB when sequences are capped at 768 tokens. Effective batch is 2 GPUs x 4 x 2 = 16 records.

Train is about 18,000 rows. Two epochs is the run below. Mid-epoch eval uses 400 val rows. The full val file runs at the end of each epoch. Checkpoints land in `/kaggle/working/soro_kev_t4`.

In [ ]:
import os
from pathlib import Path
nproc = 2 if __import__('torch').cuda.device_count() >= 2 else 1
out = Path('/kaggle/working/soro_kev_t4') if Path('/kaggle/working').exists() else Path('runs/soro_kev_t4')
cmd = (
    f'torchrun --nproc_per_node={nproc} training/train_soro_kev.py '
    f'--profile kaggle_t4 --base-model Qwen/Qwen3-0.6B '
    f'--batch-size 4 --grad-accum 2 --epochs 2 --lr 2e-4 --eval-every 400 '
    f'--data-dir "{DATA}" --output-dir "{out}"'
)
print(cmd)
rc = os.system(cmd)
print('exit', rc)

## Conformal eval

Run this after training. It calibrates on the first slice of each file and scores the rest. Alpha 0.10 means a 90% coverage target. The product read of the set size is: 1 execute, 2 to 8 ask, 0 abstain. The script reports coverage and mean set size. It does not apply that three-way rule by itself.

In [ ]:
import os
from pathlib import Path
out = Path('/kaggle/working/soro_kev_t4') if Path('/kaggle/working').exists() else Path('runs/soro_kev_t4')
reports = Path('/kaggle/working/reports') if Path('/kaggle/working').exists() else Path('reports')
reports.mkdir(parents=True, exist_ok=True)
for split, cal in [('val_zeroshot.jsonl', 2000), ('val_external_human.jsonl', 1500)]:
    cmd = (
        f'python training/evaluate_benchmark.py --backend neural --checkpoint "{out}" '
        f'--val-data "{DATA / split}" --cal-size {cal} --alpha 0.10 '
        f'--output-report "{reports / (split + ".neural.json")}"'
    )
    print(cmd)
    print('exit', os.system(cmd))